# Worked Capstone: Customer Churn Classification and Decision Thresholding

**Domain:** Imbalanced classification  
**Primary dataset:** `customer_churn.csv`  
**Level:** Practitioner to Advanced

## Business goal

Rank customers for retention review under limited outreach capacity while evaluating probability quality and subgroup behaviour.

This is a worked reference project. First attempt the corresponding phase project independently; then use this capstone to compare framing, evaluation, code structure, and communication.

## Decision questions

        1. Does the model improve over prevalence?
2. What operating threshold meets capacity/recall needs?
3. Are probabilities calibrated?
4. Which slices fail?

        ## Definition of done

        - [ ] Pipeline
- [ ] Cross-validation
- [ ] ROC/PR metrics
- [ ] Threshold policy
- [ ] Calibration table
- [ ] Slice metrics
- [ ] Saved artifact

## End-to-end workflow

```text
Decision and scope
      ↓
Data contract and quality
      ↓
Exploration and hypotheses
      ↓
Baseline and evaluation design
      ↓
Candidate method(s)
      ↓
Held-out / temporal evaluation
      ↓
Error, slice, and sensitivity analysis
      ↓
Artifacts, limitations, recommendation
```

At every stage, distinguish calculation correctness, statistical validity, operational validity, and decision validity.

## Risk register

        | Risk | Mitigation |
        |---|---|
        | Prediction confused with intervention effect | Use the score for prioritization; test retention actions separately. |
| Threshold fixed at 0.5 | Choose from cost/capacity and validate. |
| Historical label/process bias | Audit collection and subgroup performance. |

In [ ]:
from pathlib import Path
import sys
import json
import warnings
warnings.filterwarnings("ignore")

_candidates = [Path.cwd(), *Path.cwd().parents]
COURSE_ROOT = next((p for p in _candidates if (p / "datasets").exists()), Path.cwd())
DATA_DIR = COURSE_ROOT / "datasets"
ARTIFACT_DIR = COURSE_ROOT / "artifacts"
ARTIFACT_DIR.mkdir(exist_ok=True)
sys.path.insert(0, str(COURSE_ROOT))

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import display

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
print(f"Course root: {COURSE_ROOT}")

## 1. Data and holdout

Use a stratified split and exclude the identifier from features.

In [ ]:
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (roc_auc_score,average_precision_score,precision_recall_curve,
                             confusion_matrix,classification_report,brier_score_loss)
import joblib

df=pd.read_csv(DATA_DIR/"customer_churn.csv")
X=df.drop(columns=["customer_id","churn"]); y=df.churn
Xdev,Xtest,ydev,ytest=train_test_split(X,y,test_size=.2,stratify=y,random_state=42)
print("Development/test prevalence:",ydev.mean(),ytest.mean())

## 2. Pipeline and cross-validation

All learned transformations occur inside the folds.

In [ ]:
num=X.select_dtypes(include="number").columns
cat=X.select_dtypes(exclude="number").columns
pipeline=Pipeline([
    ("prep",ColumnTransformer([
        ("num",Pipeline([("impute",SimpleImputer(strategy="median")),("scale",StandardScaler())]),num),
        ("cat",Pipeline([("impute",SimpleImputer(strategy="most_frequent")),
                         ("encode",OneHotEncoder(handle_unknown="ignore"))]),cat),
    ])),
    ("model",LogisticRegression(max_iter=1000,class_weight="balanced")),
])
cv=StratifiedKFold(5,shuffle=True,random_state=42)
cv_result=cross_validate(pipeline,Xdev,ydev,cv=cv,scoring=["roc_auc","average_precision","neg_brier_score"])
display(pd.DataFrame(cv_result)[["test_roc_auc","test_average_precision","test_neg_brier_score"]].describe())

## 3. Holdout ranking and threshold

Choose an operating point satisfying recall while maximizing precision. In practice, also encode outreach capacity and costs.

In [ ]:
pipeline.fit(Xdev,ydev)
prob=pipeline.predict_proba(Xtest)[:,1]
print("ROC-AUC:",roc_auc_score(ytest,prob))
print("PR-AUC:",average_precision_score(ytest,prob),"Prevalence baseline:",ytest.mean())
precision,recall,thresholds=precision_recall_curve(ytest,prob)
operating=pd.DataFrame({"threshold":np.r_[thresholds,1.0],"precision":precision,"recall":recall})
chosen=operating[operating.recall>=.80].sort_values(["precision","threshold"],ascending=False).iloc[0]
display(chosen.to_frame("chosen"))
pred=(prob>=chosen.threshold).astype(int)
print(confusion_matrix(ytest,pred))
print(classification_report(ytest,pred,digits=3))

## 4. Calibration and slices

Group predicted scores into bins and evaluate meaningful categorical slices.

In [ ]:
calibration=pd.DataFrame({"y":ytest.to_numpy(),"prob":prob})
calibration["bin"]=pd.qcut(calibration.prob,10,duplicates="drop")
calibration_table=calibration.groupby("bin",observed=True).agg(
    n=("y","size"),mean_probability=("prob","mean"),event_rate=("y","mean")
)
display(calibration_table.round(3))
print("Brier score:",brier_score_loss(ytest,prob))

slice_frame=Xtest.copy()
slice_frame["y"]=ytest.to_numpy(); slice_frame["pred"]=pred
slice_metrics=slice_frame.groupby("contract_type").apply(
    lambda g: pd.Series({
        "n":len(g),
        "prevalence":g.y.mean(),
        "selection_rate":g.pred.mean(),
        "recall":((g.y==1)&(g.pred==1)).sum()/max((g.y==1).sum(),1),
    }),include_groups=False
)
display(slice_metrics.round(3))

## 5. Save decision artifact

Persist the model together with the selected threshold and scope statement.

In [ ]:
artifact=ARTIFACT_DIR/"capstone_churn_pipeline.joblib"
joblib.dump(pipeline,artifact)
policy={
    "artifact":str(artifact),
    "threshold":float(chosen.threshold),
    "intended_use":"Prioritize customers for human-reviewed retention outreach.",
    "excluded_use":"Do not infer that outreach will cause retention; do not use for adverse eligibility decisions.",
    "holdout_roc_auc":float(roc_auc_score(ytest,prob)),
    "holdout_pr_auc":float(average_precision_score(ytest,prob)),
}
(ARTIFACT_DIR/"capstone_churn_policy.json").write_text(json.dumps(policy,indent=2))
print(json.dumps(policy,indent=2))

## Model/project card

Complete this before presenting the result:

| Field | Statement |
|---|---|
| Intended use | |
| Excluded use | |
| Data population and coverage | |
| Target/metric definition | |
| Evaluation split | |
| Baseline | |
| Primary result | |
| Known limitations | |
| Important subgroup behaviour | |
| Human review / abstention | |
| Monitoring | |
| Owner and review cadence | |

## Final reflection

1. Which result changed your initial belief?
2. Which assumption creates the largest residual risk?
3. What simpler alternative was competitive?
4. What evidence is still required before an operational decision?
5. What would you monitor first after release?

Re-run the notebook from a clean kernel and verify generated artifacts before considering the capstone complete.